In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 1: Detección y clasificación de outliers (error de sensor vs evento real)
para datos horarios de calidad del aire.

Ejecutar en VSCode / Jupyter / terminal con Python 3.11.
Dependencias: pandas, numpy, scikit-learn, statsmodels
Instalación rápida: pip install pandas numpy scikit-learn statsmodels
"""

import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from statsmodels.tsa.seasonal import STL

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# -------------------------------------------------------------------
# PARÁMETROS CONFIGURABLES (ajustados a tus especificaciones)
# -------------------------------------------------------------------
PERIOD_H = 24                      # periodo estacional (horario → 24h)
STL_K_MAD = 4.0                    # umbral en MAD para residuo STL
JUMP_K_MAD = 6.0                   # umbral para saltos 1h basados en MAD local
ISO_CONTAMINATION = 0.01           # proporción esperada de outliers para IsolationForest
ROLL_WINDOW_H = 24                 # ventana (horas) para calcular MAD local

# Límites físicos (según tus respuestas)
PHYS_TEMP_MIN = -40.0
PHYS_TEMP_MAX = 50.0
PHYS_RSOL_MIN = 0.0
PHYS_RSOL_MAX = 1200.0
PHYS_WIND_MIN = 0.0
PHYS_WIND_MAX = 60.0
ANGLE_MIN = 0.0
ANGLE_MAX = 360.0
CONTAMINANTE_MAX = 800.0           # para NO, NO2, NOx, O3 (µg/m³)

# Carpeta de salida (se crea si no existe)
OUTPUT_DIR = os.path.expanduser("~/Documents/GitHub/TFGFinal/Datos_TFG_outliers")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------------------------------------------------------------------
# RUTAS DE LAS ESTACIONES (tal como las proporcionaste)
# -------------------------------------------------------------------
station_paths = [
    r"/usu/snsaetor/TFG_BENJA/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E1_Alicante.csv",
    r"/usu/snsaetor/TFG_BENJA/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E2_Elda.csv",
    r"/usu/snsaetor/TFG_BENJA/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E1_Elche.csv",
    r"/usu/snsaetor/TFG_BENJA/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E2_Elda.csv",
    r"/usu/snsaetor/TFG_BENJA/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E1_Valencia.csv",
    r"/usu/snsaetor/TFG_BENJA/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E2_Buñol.csv",
    r"/usu/snsaetor/TFG_BENJA/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E1_Valencia.csv",
    r"/usu/snsaetor/TFG_BENJA/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E2_Villar_Arzobispo.csv",
    r"/usu/snsaetor/TFG_BENJA/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E1_Castellon.csv",
    r"/usu/snsaetor/TFG_BENJA/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E2_Onda.csv",
    r"/usu/snsaetor/TFG_BENJA/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E1_Sant_Jordi.csv",
    r"/usu/snsaetor/TFG_BENJA/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E2_Coratxa.csv",
    r"/usu/snsaetor/TFG_BENJA/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E3_Zorita.csv",
    r"/usu/snsaetor/TFG_BENJA/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E1_Sant_Jordi.csv",
    r"/usu/snsaetor/TFG_BENJA/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E2_Morella.csv",
    r"/usu/snsaetor/TFG_BENJA/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E3_Zorita.csv",
]
# Expandir '~' a la ruta completa del usuario
station_paths = [os.path.expanduser(p) for p in station_paths]

# -------------------------------------------------------------------
# FUNCIONES AUXILIARES
# -------------------------------------------------------------------
def robust_mad(x):
    """MAD: median absolute deviation (robusto)."""
    x = np.asarray(x)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    return mad if mad > 0 else np.nanmean(np.abs(x - med))  # fallback

def ensure_datetime_index(df, datetime_col=None, freq='h'):
    """
    Convierte la columna datetime a índice DatetimeIndex, reindexa a frecuencia horaria
    (dejando NaN en los huecos) y ordena.
    """
    if datetime_col is None:
        # Buscar columna típica de fecha
        candidates = [c for c in df.columns if c.lower() in ("datetime", "fecha", "time", "timestamp")]
        if not candidates:
            raise ValueError("No se encontró columna datetime. Especifica datetime_col.")
        datetime_col = candidates[0]
    df = df.copy()
    df[datetime_col] = pd.to_datetime(df[datetime_col], errors='coerce')
    df = df.set_index(datetime_col).sort_index()
    # Rellenar para que no falten horas
    full_idx = pd.date_range(start=df.index.min(), end=df.index.max(), freq='h')
    df = df.reindex(full_idx)
    df.index.name = 'datetime'
    return df

def physical_checks(df):
    """
    Marca como error físico aquellos valores fuera de los rangos definidos.
    Devuelve DataFrame con columnas 'physical_error' (bool) y 'physical_reason' (str).
    """
    flags = pd.DataFrame(index=df.index)
    flags['physical_error'] = False
    flags['physical_reason'] = ""

    # Radiación solar
    if 'R.Sol.' in df.columns:
        mask = (df['R.Sol.'] < PHYS_RSOL_MIN) | (df['R.Sol.'] > PHYS_RSOL_MAX) & (~df['R.Sol.'].isna())
        flags.loc[mask, 'physical_error'] = True
        flags.loc[mask, 'physical_reason'] += "R.Sol. fuera [0,1200]; "

    # Velocidad viento
    if 'Veloc.' in df.columns:
        mask = (df['Veloc.'] < PHYS_WIND_MIN) | (df['Veloc.'] > PHYS_WIND_MAX) & (~df['Veloc.'].isna())
        flags.loc[mask, 'physical_error'] = True
        flags.loc[mask, 'physical_reason'] += "Veloc. fuera [0,60]; "

    # Dirección y Ángulo (0-360)
    for col in ['Direc.', 'Angulo']:
        if col in df.columns:
            mask = (~df[col].isna()) & ((df[col] < ANGLE_MIN) | (df[col] >= ANGLE_MAX))
            flags.loc[mask, 'physical_error'] = True
            flags.loc[mask, 'physical_reason'] += f"{col} fuera [0,360); "

    # Temperatura
    if 'Temp.' in df.columns:
        mask = (~df['Temp.'].isna()) & ((df['Temp.'] < PHYS_TEMP_MIN) | (df['Temp.'] > PHYS_TEMP_MAX))
        flags.loc[mask, 'physical_error'] = True
        flags.loc[mask, 'physical_reason'] += "Temp. fuera [-40,50]; "

    # Contaminantes (NO, NO2, NOx, O3): no negativos y < 800
    for pol in ['NO', 'NO2', 'NOx', 'O3']:
        if pol in df.columns:
            mask = (~df[pol].isna()) & ((df[pol] < 0) | (df[pol] > CONTAMINANTE_MAX))
            flags.loc[mask, 'physical_error'] = True
            flags.loc[mask, 'physical_reason'] += f"{pol} fuera [0,800]; "

    # Distancia (positiva)
    if 'Dist.' in df.columns:
        mask = (~df['Dist.'].isna()) & (df['Dist.'] < 0)
        flags.loc[mask, 'physical_error'] = True
        flags.loc[mask, 'physical_reason'] += "Dist. negativa; "

    return flags

def stl_outliers(df, col='O3', period=PERIOD_H, k=STL_K_MAD):
    """Detecta outliers mediante residuos STL + umbral MAD."""
    res = pd.Series(False, index=df.index)
    try:
        s = df[col].dropna()
        if len(s) > period * 2:
            stl = STL(s, period=period, robust=True)
            out = stl.fit()
            resid = out.resid
            mad_resid = robust_mad(resid)
            threshold = k * mad_resid
            mask = np.abs(resid) > threshold
            res.loc[mask.index] = mask.values
    except Exception as e:
        print(f"STL falló para {col}: {e}")
    return res

def rolling_jump_outliers(df, col='O3', window=ROLL_WINDOW_H, k=JUMP_K_MAD):
    """Detecta saltos bruscos de 1 hora comparando diff con MAD local."""
    res = pd.Series(False, index=df.index)
    s = df[col]
    diff = s.diff().abs()
    # MAD móvil de las diferencias
    roll_mad = diff.rolling(window=window, min_periods=1, center=True).apply(lambda x: robust_mad(x), raw=True)
    thr = k * roll_mad
    mask = (diff > thr) & (~diff.isna())
    res.loc[mask.index] = mask.values
    return res

def isolation_forest_outliers(df, features=None, contamination=ISO_CONTAMINATION):
    """Isolation Forest multivariado sobre las variables numéricas disponibles."""
    if features is None:
        candidates = ['O3', 'NO', 'NO2', 'NOx', 'Temp.', 'Veloc.', 'R.Sol.']
        features = [c for c in candidates if c in df.columns]
    res = pd.Series(False, index=df.index)
    sub = df[features].copy()
    valid_idx = sub.dropna(how='all').index
    if len(valid_idx) < 20:
        return res
    sub = sub.loc[valid_idx]
    try:
        # Relleno simple para poder correr IsolationForest
        sub = sub.ffill().bfill().fillna(0)
        iso = IsolationForest(contamination=contamination, random_state=RANDOM_STATE)
        pred = iso.fit_predict(sub.values)
        out_idx = sub.index[pred == -1]
        res.loc[out_idx] = True
    except Exception as e:
        print("IsolationForest falló:", e)
    return res

def neighbor_support(df, idx, col='O3', window=3, rel_tol=0.5):
    """
    Evalúa si un valor atípico tiene apoyo en sus vecinos temporales.
    Retorna True si el cambio respecto a la mediana vecina es pequeño en términos relativos.
    """
    half = (window - 1) // 2
    i_pos = df.index.get_loc(idx)
    start = max(0, i_pos - half)
    end = min(len(df) - 1, i_pos + half)
    vals = df[col].iloc[start:end+1].values
    center = df[col].iloc[i_pos]
    if np.isnan(center):
        return False
    if np.isnan(vals).sum() >= half + 1:
        return False
    med_neighbors = np.nanmedian(np.concatenate([vals[:half], vals[half+1:]])) if len(vals) > 1 else np.nan
    if np.isnan(med_neighbors):
        return False
    # Si la diferencia con la mediana vecina es menor que el valor absoluto * rel_tol, hay soporte
    if abs(center - med_neighbors) <= abs(center) * rel_tol:
        return True
    return False

def related_variables_spike(df, idx, variables, window=ROLL_WINDOW_H, k=2.0):
    """
    Comprueba si alguna de las variables relacionadas presenta también un pico
    (valor > k * MAD local) en el mismo instante.
    """
    i_pos = df.index.get_loc(idx)
    for var in variables:
        if var not in df.columns:
            continue
        val = df[var].iloc[i_pos]
        if pd.isna(val):
            continue
        # Mediana móvil centrada
        roll_med = df[var].rolling(window=window, min_periods=1, center=True).median()
        med = roll_med.iloc[i_pos]
        if pd.isna(med):
            continue
        mad = robust_mad(df[var].dropna())
        if mad == 0:
            continue
        if abs(val - med) > k * mad:
            return True
    return False

# -------------------------------------------------------------------
# PROCESAMIENTO DE UNA ESTACIÓN
# -------------------------------------------------------------------
def process_station(path, save_out=True, verbose=True):
    """
    Carga el CSV, aplica detección de outliers y clasifica en:
    - sensor_error: error físico o aislado sin apoyo vecinal.
    - evento_real: múltiples detectores y picos en variables relacionadas.
    - doubtful: casos ambiguos.
    Guarda un CSV con sufijo '_outliers' y devuelve el DataFrame procesado.
    """
    station_name = Path(path).stem
    if verbose:
        print(f"\nProcesando: {station_name}")

    # Cargar y estandarizar índice temporal
    df_raw = pd.read_csv(path)
    df = ensure_datetime_index(df_raw, datetime_col=None, freq='h')

    # Inicializar flags
    flags = pd.DataFrame(index=df.index)
    flags['physical_error'] = False
    flags['physical_reason'] = ""

    # 1. Comprobaciones físicas
    phys = physical_checks(df)
    flags['physical_error'] = phys['physical_error']
    flags['physical_reason'] = phys['physical_reason']

    # 2. Detección estadística (sobre O3 si existe)
    o3_col = 'O3' if 'O3' in df.columns else None
    if o3_col is None:
        print(f"  Aviso: {station_name} no tiene columna O3. Se saltan STL/jump/iso para O3.")
        stl_mask = pd.Series(False, index=df.index)
        jump_mask = pd.Series(False, index=df.index)
    else:
        stl_mask = stl_outliers(df, col=o3_col, period=PERIOD_H, k=STL_K_MAD)
        jump_mask = rolling_jump_outliers(df, col=o3_col, window=ROLL_WINDOW_H, k=JUMP_K_MAD)

    flags['outlier_stl'] = stl_mask
    flags['outlier_jump'] = jump_mask

    # 3. Isolation Forest multivariado (usa todas las variables numéricas)
    iso_mask = isolation_forest_outliers(df, features=None, contamination=ISO_CONTAMINATION)
    flags['outlier_iso'] = iso_mask

    # 4. Clasificación heurística: sensor_error / evento_real / doubtful
    flags['sensor_error'] = False
    flags['evento_real'] = False
    flags['doubtful'] = False

    related_vars = ['NO', 'NO2', 'NOx', 'Temp.', 'R.Sol.', 'Veloc.']

    for ts in df.index:
        # Si ya es error físico, se marca directamente como sensor_error
        if flags.at[ts, 'physical_error']:
            flags.at[ts, 'sensor_error'] = True
            continue

        is_stl = flags.at[ts, 'outlier_stl']
        is_jump = flags.at[ts, 'outlier_jump']
        is_iso = flags.at[ts, 'outlier_iso']
        any_det = is_stl or is_jump or is_iso
        if not any_det:
            continue

        n_det = sum([is_stl, is_jump, is_iso])

        # Soporte en vecinos (solo para O3)
        support = False
        if o3_col is not None and not pd.isna(df.at[ts, o3_col]):
            try:
                support = neighbor_support(df, ts, col=o3_col, window=3, rel_tol=0.6)
            except Exception:
                support = False

        # Picos en variables relacionadas
        spike_related = related_variables_spike(df, ts, related_vars, window=ROLL_WINDOW_H, k=2.0)

        # Reglas de decisión
        if n_det >= 2 and spike_related:
            flags.at[ts, 'evento_real'] = True
        else:
            if not support and n_det >= 1:
                flags.at[ts, 'sensor_error'] = True
            else:
                flags.at[ts, 'doubtful'] = True

    # 5. Preparar columna para imputación futura: O3_for_impute = O3 salvo donde sensor_error = True (se pone NaN)
    df_proc = df.copy()
    if o3_col is not None:
        df_proc['O3_for_impute'] = df_proc[o3_col].where(~flags['sensor_error'], np.nan)

    # Unir flags al DataFrame
    out_df = pd.concat([df_proc, flags], axis=1)

    # Guardar CSV
    if save_out:
        out_path = os.path.join(OUTPUT_DIR, f"{station_name}_outliers.csv")
        out_df.to_csv(out_path, index=True)
        if verbose:
            print(f"  Guardado: {out_path}")

    return out_df

# -------------------------------------------------------------------
# EJECUCIÓN SOBRE TODAS LAS ESTACIONES
# -------------------------------------------------------------------
all_doubtful = []
all_summaries = []

for path in station_paths:
    try:
        df_out = process_station(path, save_out=True, verbose=True)

        # Recopilar filas dudosas (doubtful)
        if 'doubtful' in df_out.columns:
            doubtful_rows = df_out[df_out['doubtful'] == True].copy()
            if not doubtful_rows.empty:
                doubtful_rows['station_file'] = Path(path).stem
                all_doubtful.append(doubtful_rows)

        # Resumen de conteos
        summary = {
            'station': Path(path).stem,
            'n_rows': len(df_out),
            'n_physical_error': int(df_out['physical_error'].sum()) if 'physical_error' in df_out.columns else 0,
            'n_outlier_stl': int(df_out['outlier_stl'].sum()) if 'outlier_stl' in df_out.columns else 0,
            'n_outlier_jump': int(df_out['outlier_jump'].sum()) if 'outlier_jump' in df_out.columns else 0,
            'n_outlier_iso': int(df_out['outlier_iso'].sum()) if 'outlier_iso' in df_out.columns else 0,
            'n_sensor_error': int(df_out['sensor_error'].sum()) if 'sensor_error' in df_out.columns else 0,
            'n_evento_real': int(df_out['evento_real'].sum()) if 'evento_real' in df_out.columns else 0,
            'n_doubtful': int(df_out['doubtful'].sum()) if 'doubtful' in df_out.columns else 0,
        }
        all_summaries.append(summary)

    except Exception as e:
        print(f"ERROR procesando {path}: {e}")

# Guardar casos dudosos combinados
if all_doubtful:
    combined_doubtful = pd.concat(all_doubtful, axis=0)
    doubtful_path = os.path.join(OUTPUT_DIR, "doubtful_cases_combined.csv")
    combined_doubtful.to_csv(doubtful_path, index=True)
    print(f"\nCasos dudosos combinados guardados en: {doubtful_path}")

# Guardar resumen global
if all_summaries:
    summary_df = pd.DataFrame(all_summaries)
    summary_path = os.path.join(OUTPUT_DIR, "outliers_summary.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"Resumen global guardado en: {summary_path}")

print("\nProceso completado. Revisa la carpeta:", OUTPUT_DIR)